# Inteligência Artificial 2025/2026
## Trabalho: Adversarial Search (MCTS) e Árvores de Decisão (ID3)
### Jogo: PopOut (variante do Connect-4)

---

1. **O Jogo PopOut** — regras e implementação do motor de jogo  
2. **Monte Carlo Tree Search (MCTS)** — implementação, variantes e benchmark  
3. **Geração do Dataset** — geração de pares (estado, movimento) com resultados reais  
4. **Árvore de Decisão ID3 — Iris** — warm-up com dataset contínuo, resultados e análise  
5. **Árvore de Decisão ID3 — PopOut** — resultados reais, matriz de confusão e análise  
6. **Modos de Jogo** — Human vs Human, Human vs MCTS, MCTS vs ID3  
7. **Conclusões**


---
## 1. Configuração e Imports

In [ ]:
import math
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from IPython.display import Image, display

print('Imports OK')

---
## 2. O Jogo PopOut

PopOut é uma variante do Connect-4 num tabuleiro 6x7. O objetivo é o mesmo — alinhar 4 peças consecutivas na horizontal, vertical ou diagonal. A grande diferença está na possibilidade de **retirar (_pop_) a própria peça da base** de qualquer coluna onde o jogador tenha uma peça na linha inferior. Ao retirar, todas as peças acima descem uma posição.

### Regras adicionais face ao Connect-4:
1. **Pop simultâneo:** se um pop cria 4 em linha para ambos os jogadores, o jogador que fez o pop ganha.
2. **Tabuleiro cheio:** o jogador a mover pode declarar empate em vez de jogar.
3. **Repetição:** se o mesmo estado se repetir 3 vezes, qualquer jogador pode declarar empate.

Estas regras tornam o PopOut substancialmente mais complexo que o Connect-4: o espaço de estados é maior (os pops criam ciclos) e a avaliação posicional é mais difícil.

In [ ]:
from PopOut4 import (
    ROWS, COLS, EMPTY, P1, P2,
    create_board, print_board, board_to_tuple,
    check_four, get_moves, drop_disc, pop_disc,
    other, is_full, can_drop, can_pop
)

print(f'Tabuleiro: {ROWS} linhas x {COLS} colunas')
print(f'Jogadores: {P1} (X) e {P2} (O)')

board = create_board()
board = drop_disc(board, 3, P1)
board = drop_disc(board, 3, P2)
board = drop_disc(board, 2, P1)
board = drop_disc(board, 3, P1)
print('\nExemplo de tabuleiro com algumas jogadas:')
print_board(board)

drops, pops = get_moves(board, P1)
print(f'Movimentos disponiveis para X:')
print(f'  Drop: colunas {[c+1 for c in drops]}')
print(f'  Pop : colunas {[c+1 for c in pops]}')

---
## 3. Monte Carlo Tree Search (MCTS)

### 3.1 Visão Geral

O MCTS é um algoritmo de pesquisa adversarial baseado em simulações de Monte Carlo. Em vez de explorar exaustivamente a árvore de jogo (inviável no PopOut pelo enorme fator de ramificação e pelos ciclos introduzidos pelos pops), constrói progressivamente uma árvore guiada por estatísticas de visitas e vitórias.

Cada iteração tem **4 fases**:

| Fase | Descrição |
|------|----------|
| **Seleção** | Desce a árvore usando UCB, escolhendo o filho com maior score até encontrar um nó não totalmente expandido |
| **Expansão** | Adiciona um novo filho correspondente a um movimento ainda não tentado |
| **Rollout** | Simula um jogo completo (aleatório ou guiado) a partir do novo nó |
| **Backpropagation** | Propaga o resultado de volta até à raiz, atualizando visitas e wins de cada nó percorrido |

No final das N iterações, o movimento escolhido é o **filho mais visitado** — esta escolha é mais robusta do que escolher o filho com maior win-rate, pois tem menor variância.

### 3.2 Variantes UCB implementadas

**UCT (Upper Confidence Bound for Trees)** — variante clássica (Kocsis & Szepesvári, 2006):

$$\text{UCT}(v) = \frac{w_i}{n_i} + C \cdot \sqrt{\frac{\ln N}{n_i}}$$

onde $w_i$ são as vitórias do filho, $n_i$ as suas visitas, $N$ as visitas do pai, e $C$ a constante de exploração.

**UCB-V** — variante consciente da variância (Audibert et al., 2009):

$$\text{UCB-V}(v) = \frac{w_i}{n_i} + \sqrt{\frac{2 \cdot \text{Var}_i \cdot \ln N}{n_i}} + C \cdot \frac{\ln N}{n_i}$$

### 3.3 Extensões implementadas

**`use_heuristic`** — adiciona um bónus posicional $H(s,a)/(n+1)$ ao score UCB. O bónus valoriza colunas centrais (que participam em mais linhas de 4), atribui +1.0 a movimentos que ganham imediatamente e +0.7 a movimentos que bloqueiam vitória imediata do adversário. O denominador $(n+1)$ é deliberado: garante que o bónus **desvanece com as visitas**, dominando nas primeiras explorações mas tendendo para zero à medida que o nó acumula experiência empírica — o conhecimento de domínio guia sem nunca substituir as estatísticas reais.

**`smart_rollout`** — rollout guiado por conhecimento de domínio (Heavy Playouts):

Em vez de simular jogadas completamente aleatórias, o smart rollout aplica uma política simples durante a fase de simulação:
```
Prioridade: vitória imediata > bloquear vitória adversária > coluna central > aleatório
```
A motivação para usar smart rollout vem da literatura de MCTS para jogos de conexão (Gelly & Silver, 2007; Chaslot et al., 2008): rollouts mais realistas produzem estimativas de valor mais precisas, reduzindo o número de simulações necessárias para convergir. No entanto, o seu efeito depende da qualidade da política de rollout e do orçamento disponível — investigado experimentalmente no torneio.

**Escolha do UCB-V como alternativa ao UCT:**

O UCB-V (Audibert et al., 2009) foi escolhido por razões teóricas bem fundamentadas. O UCT assume implicitamente que as recompensas têm variância máxima ($\text{Var} = 0.25$ para recompensas em $[0,1]$). No PopOut, com smart rollout, os rollouts tendem a ser mais consistentes — a variância real é inferior ao máximo. O UCB-V incorpora a **variância amostral real**:
$$\text{Var}_i = \frac{\sum r^2}{n_i} - \left(\frac{\sum r}{n_i}\right)^2$$
Quando a variância é baixa (smart rollout consistente), o UCB-V explora menos e explota mais — aproveitando o facto de que os rollouts são informativos. Esta é a base teórica para combinar UCB-V com smart rollout: as duas extensões **complementam-se mutuamente**.

**`max_children`** — limita o número de filhos expandidos por nó, reduzindo o branching factor e concentrando o orçamento nos movimentos mais promissores.

In [ ]:
from MCTS import MCTS

agent = MCTS(c=1.41, variant='ucbv', use_heuristic=False, smart_rollout=True, n_simulations=300)

board = create_board()
t0 = time.perf_counter()
move = agent.choose_move(board, P1)
dt = time.perf_counter() - t0

print(f'Melhor movimento para X no tabuleiro inicial: {move[0]} coluna {move[1]+1}')
print(f'Tempo de decisao: {dt:.2f}s com 300 simulacoes')
print()
print('Num tabuleiro vazio, o MCTS converge para a coluna central (col 4)')
print('pois esta coluna faz parte do maior numero de linhas de 4 possiveis.')

### 3.4 Torneio de Configurações MCTS

Para identificar a configuração mais forte, realizámos um torneio round-robin entre três configurações, com 150 jogos por par (450 no total). Em cada par os jogos alternaram qual o agente começa, eliminando o viés de primeiro jogador. As configurações testadas foram o UCT padrão (c=√2, sem extensões), o UCT com bónus posicional (c=1.41), e o UCB-V combinado com smart rollout.

In [ ]:
display(Image('mcts_tournament.png'))

### 3.5 Análise dos Resultados do Torneio

**Configuração do torneio:** 3 configurações, 150 jogos por par, 450 partidas no total. Cada par alterna quem começa para eliminar o viés de primeiro jogador.

**Confrontos detalhados:**

| Par | Resultado | Win-rate |
|-----|-----------|----------|
| Default (UCT) vs UCT + Heurística | 72W / 77L / 1D | 48% |
| Default (UCT) vs UCB-V + Smart Rollout | 36W / 114L / 0D | 24% |
| UCT + Heurística vs UCB-V + Smart Rollout | 27W / 122L / 1D | 18% |

**Ranking Final (win-rate médio contra todos os adversários):**

| Posição | Configuração | Win-rate médio | Tempo/decisão |
|---------|-------------|---------------|---------------|
| 🥇 1.º | **UCB-V + Smart Rollout** | **78.7%** | 1.793s |
| 🥈 2.º | Default (UCT) | 36.0% | 0.445s |
| 🥉 3.º | UCT + Heurística | 34.7% | 1.685s |

**Interpretação:**

O resultado mais surpreendente é a **vitória clara do UCB-V + Smart Rollout** (78.7%), dominando ambos os adversários — 76% contra o UCT simples e 81% contra o UCT + Heurística. Este resultado contraria a intuição de que rollouts guiados prejudicam a exploração.

A explicação para a supremacia do UCB-V + Smart Rollout neste torneio:
- O **smart rollout** guia as simulações para movimentos mais realistas (vitória imediata, bloqueio), produzindo estimativas de valor mais precisas por simulação
- O **UCB-V** incorpora a variância dos resultados — quando o smart rollout torna as simulações mais consistentes (menos aleatórias), a variância baixa e o UCB-V consegue distinguir melhor entre boas e más jogadas
- Em conjunto, as duas extensões complementam-se: o smart rollout reduz o ruído, e o UCB-V aproveita essa redução de ruído para explorar mais eficientemente

O **UCT + Heurística** ficou em último lugar (34.7%), abaixo do UCT simples (36%). A heurística posicional, apesar de parecer benéfica, **aumenta drasticamente o tempo por decisão** (1.685s vs 0.445s do UCT simples) — com o mesmo orçamento de tempo, o UCT simples consegue fazer mais simulações e explorar melhor a árvore.

**Decisão:** para a geração do dataset e para jogar usamos **UCB-V + Smart Rollout (c=1.41)** — o vencedor do torneio com 78.7% de win-rate. O smart rollout produz decisões mais realistas e consistentes, tornando-o o melhor oráculo possível para gerar labels de treino de qualidade para o ID3.

---
## 4. Geração do Dataset

### 4.1 Motivação e estratégia

Para treinar o ID3 no PopOut, precisamos de pares **(estado do jogo → melhor movimento)**. O MCTS funciona como oráculo: para cada posição decide o melhor movimento com 300 simulações, e essa decisão é registada como exemplo de treino.

**Representação do estado (43 features):**
- `cell_r_c` (42 colunas): valor de cada célula — `'X'`, `'O'`, ou `'.'`
- `player`: de quem é a vez — `'X'` ou `'O'`

**Label:** `drop_N` ou `pop_N` (N = coluna em índice base 0).

### 4.2 Ciclo de geração

| Modo (game % 4) | Configuração | Objetivo |
|----------------|-------------|----------|
| 0 | MCTS(X) vs Random(O) | MCTS vê posições criadas por adversário caótico |
| 1 | Random(X) vs MCTS(O) | Idem, MCTS como O |
| 2 | MCTS(X) vs MCTS(O) | Posições de qualidade, ambos registados |
| 3 | Random(X) vs MCTS(O) | Reforço de posições variadas |

Só os movimentos do MCTS são guardados. O CSV é guardado a cada 50 jogos — proteção contra crash.

### 4.3 Configuração e resultados da corrida

```
Duração:          8 horas
Simulações/mov:   300
Agente MCTS:      UCB-V, c=1.41, use_heuristic=False, smart_rollout=True
Guardar CSV:      a cada 50 jogos (protecção contra crash)
Velocidade:       ~1159 jogos/hora (estável ao longo das 8h)

Total jogos:      9267
Total amostras:   77101
Win-rate MCTS:    90.7%  (W:8402  L:863  D:2)

drop_3    20619    pop_4       446
drop_2    13409    pop_3       436
drop_4    13393    pop_2       415
drop_5     8963    pop_5       313
drop_1     8852    pop_1       313
drop_6     4802    pop_0       187
drop_0     4778    pop_6       175
```

**Porquê 8 horas e 300 simulações?**

O orçamento de 300 simulações por movimento foi escolhido como compromisso entre qualidade das decisões e velocidade de geração. Com 300 simulações, o MCTS atinge ~90% de win-rate contra adversário aleatório — indicador de que as decisões são de qualidade suficiente para servir como labels de treino. Aumentar para 1000 simulações melhoraria ligeiramente a qualidade mas reduziria a velocidade para ~350 jogos/hora, gerando apenas ~17500 amostras em 8 horas — insuficiente para treinar um ID3 com 14 classes e 43 features.

As 8 horas foram escolhidas para maximizar o volume de dados dentro do tempo disponível, atingindo 77101 amostras — quantidade razoável para um ID3 mas ainda com subrepresentação das classes pop (ver análise na secção 6).

In [ ]:
df = pd.read_csv('popout_dataset.csv')
print(f'Dataset: {len(df)} amostras, {len(df.columns)-1} features + label')
print(f'Movimentos unicos: {df["move"].nunique()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

move_counts = df['move'].value_counts()
colors = ['#2196F3' if 'drop' in m else '#FF5722' for m in move_counts.index]
move_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Distribuicao de movimentos no dataset', fontweight='bold')
axes[0].set_xlabel('Movimento')
axes[0].set_ylabel('Frequencia')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(handles=[
    mpatches.Patch(color='#2196F3', label='drop'),
    mpatches.Patch(color='#FF5722', label='pop')
], loc='upper right')

move_types = df['move'].str.split('_').str[0].value_counts()
move_types.plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                colors=['#2196F3', '#FF5722'], startangle=90)
axes[1].set_title('Drop vs Pop no dataset', fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('dataset_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.4 Análise do dataset

**Velocidade estável:** ~1159 jogos/hora ao longo das 8h — o MCTS tem custo constante com 300 simulações fixas.

**Win-rate 90.7%:** confirma a superioridade do MCTS sobre o adversário aleatório.

**Desequilíbrio drop vs pop (~33:1):** não é erro — reflete a realidade do jogo. No início o tabuleiro está vazio (não há peças na base para fazer pop) e na maioria das posições intermédias um drop é estrategicamente superior. Este desequilíbrio terá impacto direto no ID3.

**Preferência pela coluna central:** `drop_3` tem 20619 amostras, quase o dobro de `drop_2` e `drop_4`. O MCTS aprende que a coluna central (col 4 em base 1) faz parte do maior número de linhas de 4 possíveis.

**Simetria:** `drop_2`/`drop_4` (13409 vs 13393) e `drop_1`/`drop_5` (8852 vs 8963) têm frequências muito próximas — o MCTS não tem viés de posição.

---
## 5. Árvore de Decisão ID3 — Dataset Iris

### 5.1 Objetivo e desafio

O Iris serve de warm-up para validar a implementação do ID3. O desafio principal é que o ID3 original (Quinlan, 1986) trata apenas atributos **categóricos**, mas o Iris tem 4 atributos **numéricos contínuos** que precisam de ser discretizados.

### 5.2 Decisões de implementação

| Componente | Escolha | Justificação |
|-----------|---------|-------------|
| Critério de split | **Gain Ratio** | O Information Gain puro favorece atributos com muitos valores. O Gain Ratio normaliza pela entropia do split |
| Discretização | **MDL (Fayyad & Irani, 1993)** | Encontra automaticamente o número mínimo de cortes que reduzem a entropia. Evita overfitting |
| Poda | **Reduced Error Pruning** | Usa o conjunto de validação para substituir subárvores por folhas quando isso não piora a accuracy |
| Avaliação | **K-fold estratificado (k=5)** | Com apenas 150 amostras, hold-out único tem alta variância |
| Split | **70/15/15** | Validação separada do teste para evitar data leakage na poda |

### 5.3 Porquê K-fold estratificado?

O **k-fold cross-validation** divide o dataset em k=5 partições de igual tamanho. Em cada iteração, 4 partições são usadas para treino e 1 para teste, rodando até todos os exemplos terem sido usados exatamente uma vez para teste. A accuracy final é a média das 5 iterações.

A versão **estratificada** garante que cada fold tem a mesma proporção de classes que o dataset completo — essencial com 3 classes de 50 amostras cada, pois uma divisão aleatória poderia criar folds com distribuições desequilibradas.

**Vantagem face ao hold-out simples:** com 150 amostras, um hold-out de 30% (45 amostras) tem alta variância — a accuracy pode variar vários pontos percentuais dependendo de quais amostras caem no teste. O k-fold usa **todos os dados para teste** (em diferentes iterações), produzindo uma estimativa muito mais estável e fiável da performance real.

In [ ]:
from ID3_final import (
    stratified_split, kfold_split,
    discretise, discretise_new,
    id3, prune, accuracy, predict_all,
    count_nodes, tree_depth, print_tree,
    show_tree, load_iris_csv
)

df_iris = load_iris_csv('iris.csv')
cols    = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
classes = sorted(df_iris['class'].unique())
X_iris, y_iris = df_iris[cols], df_iris['class']

Xtr, Xtmp, ytr, ytmp = stratified_split(X_iris, y_iris, test_size=0.30, seed=42)
Xv,  Xte,  yv,  yte  = stratified_split(Xtmp, ytmp, test_size=0.50, seed=42)
print(f'Split: {len(Xtr)} treino / {len(Xv)} val / {len(Xte)} teste')

Xtr_d, cuts = discretise(Xtr, cols, ytr)
Xv_d        = discretise_new(Xv,  cols, cuts)
Xte_d       = discretise_new(Xte, cols, cuts)

print('\nCortes MDL encontrados:')
for col in cols:
    print(f'  {col}: {[round(t,3) for t in cuts[col]]}')

In [ ]:
tree_iris = id3(Xtr_d, ytr, list(Xtr_d.columns))
print(f'Antes da poda : {count_nodes(tree_iris)} nos, profundidade {tree_depth(tree_iris)}, acc {accuracy(tree_iris, Xte_d, yte):.2%}')

tree_iris = prune(tree_iris, Xv_d, yv)
print(f'Depois da poda: {count_nodes(tree_iris)} nos, profundidade {tree_depth(tree_iris)}, acc {accuracy(tree_iris, Xte_d, yte):.2%}')

preds_iris = predict_all(tree_iris, Xte_d)
print('\nAcuracia por classe:')
for cls in classes:
    mask = yte == cls
    ok   = sum(p == cls for p, m in zip(preds_iris, mask) if m)
    print(f'  {cls}: {ok}/{mask.sum()}')

print('\nValidacao cruzada k-fold (k=5):')
fold_accs = []
for fold_i, (Xf_tr, yf_tr, Xf_te, yf_te) in enumerate(kfold_split(X_iris, y_iris, k=5)):
    Xf_tr_d, c = discretise(Xf_tr, cols, yf_tr)
    Xf_te_d    = discretise_new(Xf_te, cols, c)
    t = id3(Xf_tr_d, yf_tr, list(Xf_tr_d.columns))
    acc_fold = accuracy(t, Xf_te_d, yf_te)
    fold_accs.append(acc_fold)
    print(f'  Fold {fold_i+1}: {acc_fold:.2%}')
print(f'  Media: {np.mean(fold_accs):.2%}  +-{np.std(fold_accs):.2%}')

### 5.3 Resultados obtidos

```
Split: 105 treino / 21 val / 24 teste

Cortes MDL: sepal_length: [5.5]  sepal_width: [3.4]
            petal_length: [2.45, 4.8]  petal_width: [0.8, 1.8]

Antes da poda : 14 nos, profundidade 4, acc 95.83%
Depois da poda:  4 nos, profundidade 1, acc 95.83%

Acuracia por classe:
  Iris-setosa:     8/8   (100%)
  Iris-versicolor: 7/8   (87.5%)
  Iris-virginica:  8/8   (100%)

Validacao cruzada k-fold (k=5):
  Fold 1: 100.00%
  Fold 2:  96.67%
  Fold 3:  96.67%
  Fold 4:  93.33%
  Fold 5:  93.33%
  Media: 96.00%  +-2.49%
```

### 5.4 Visualização das árvores — antes e depois da poda

**Árvore antes da poda (ID3_Iris.py — 14 nós, profundidade 4):**

In [ ]:
display(Image('tree_visual.png'))

**Árvore depois da poda (ID3_final.py — 4 nós, profundidade 1):**

In [ ]:
display(Image('tree_visual_final.png'))

### 5.5 Análise detalhada dos resultados

#### Cortes MDL — interpretação

A MDL encontrou cortes muito informativos e biologicamente coerentes:
- **`petal_length <= 2.45`** separa perfeitamente a *Iris-setosa* (pétalas pequenas) de todas as outras
- **`petal_length <= 4.80`** e **`petal_width <= 1.80`** distinguem *versicolor* de *virginica*
- `sepal_length` e `sepal_width` só aparecem para resolver ambiguidades nas fronteiras — têm menor poder discriminativo

#### Efeito da poda — resultado notável

A árvore passa de **14 nós (profundidade 4)** para apenas **4 nós (profundidade 1)** mantendo exatamente a mesma accuracy de 95.83%. Isto é o resultado mais impressionante do Iris.

A árvore podada tem uma estrutura minimalista: a raiz pergunta apenas `petal_length` e divide diretamente nas 3 classes:
- `petal_length <= 2.45` → **Iris-setosa**
- `petal_length <= 4.80` → **Iris-versicolor**
- `petal_length > 4.80` → **Iris-virginica**

Os 10 nós extra da árvore original eram todos subárvores de overfitting — aprendiam padrões específicos das 105 amostras de treino que não generalizavam para o teste. A poda eliminou-os com custo zero em accuracy, demonstrando que a Reduced Error Pruning funciona corretamente.

#### A diferença entre ID3_Iris.py e ID3_final.py

O `ID3_Iris.py` não reduz os nós na poda (14 → 14), enquanto o `ID3_final.py` reduz de 14 para 4. A diferença está na implementação da poda: o `ID3_final.py` usa um conjunto de validação de 21 amostras mais eficientemente, conseguindo identificar e eliminar as subárvores desnecessárias.

#### Acurácia por classe

- **Iris-setosa: 8/8 (100%)** — separável linearmente por `petal_length <= 2.45`. Nunca é confundida.
- **Iris-versicolor: 7/8 (87.5%)** — o único erro. *Versicolor* e *virginica* têm sobreposição na fronteira `petal_length ~ 4.80`, e 1 amostra de *versicolor* com pétala longa é classificada como *virginica*.
- **Iris-virginica: 8/8 (100%)** — bem separada após o corte em 4.80.

#### K-fold: 96.00% ± 2.49%

A média de 96% com desvio padrão de 2.49% confirma boa generalização. A variância é esperada dado o tamanho reduzido do dataset (150 amostras) — partições diferentes podem incluir ou excluir as amostras na fronteira *versicolor/virginica*, afetando o resultado.

---
## 6. Árvore de Decisão ID3 — PopOut

### 6.1 Diferenças face ao Iris

| Aspeto | Iris | PopOut |
|--------|------|--------|
| Features | 4 numéricas | 43 categóricas |
| Classes | 3 | 14 (7 drops + 7 pops) |
| Discretização | MDL necessária | **Não necessária** (já categórico) |
| Amostras treino | ~105 | **53.966** |
| Balanceamento | Equilibrado | **Muito desequilibrado** (drop >> pop) |

### 6.2 Dois níveis de avaliação

- **Acurácia exata:** previsão coincide exatamente com o MCTS (`drop_3 == drop_3`)
- **Acurácia de tipo:** só verifica drop vs pop, ignorando a coluna

O segundo nível avalia a **decisão estratégica** fundamental: quando fazer pop em vez de drop.

In [ ]:
import pickle

# Carrega a arvore ja treinada pelo ID3_Treino.py
with open('id3_tree.pkl', 'rb') as f:
    tree_popout = pickle.load(f)

print('Arvore ID3 carregada do pickle.')

### 6.3 Resultados obtidos

```
Split: 53966 treino / 11564 val / 11571 teste
Treino concluido em 91.2s

Antes da poda : 65991 nos, profundidade 43, acc 30.11%
Depois da poda:  1752 nos, profundidade 41, acc 29.58%

Acuracia por movimento:
  drop_0     2/717    (0%)      pop_0      0/29    (0%)
  drop_1    22/1328   (2%)      pop_1      0/47    (0%)
  drop_2   208/2012  (10%)      pop_2      0/63    (0%)
  drop_3  2902/3093  (94%)      pop_3      0/66    (0%)
  drop_4   227/2009  (11%)      pop_4      0/67    (0%)
  drop_5    38/1345   (3%)      pop_5      1/47    (2%)
  drop_6    23/721    (3%)      pop_6      0/27    (0%)

Acuracia do tipo (drop vs pop): 11221/11571 (97.0%)
Atributo raiz: cell_0_3

Validacao cruzada k-fold (k=5):
  Fold 1: acc=30.85%  tipo=95.91%
  Fold 2: acc=30.15%  tipo=95.89%
  Fold 3: acc=29.86%  tipo=95.88%
  Fold 4: acc=29.97%  tipo=95.96%
  Fold 5: acc=29.27%  tipo=95.85%
  Media acc  : 30.02%  +-0.51%
  Media tipo : 95.90%  +-0.04%
```

### 6.4 Matriz de Confusão

In [ ]:
display(Image('confusion_matrix_popout.png'))

### 6.5 Análise detalhada dos resultados

#### Acurácia exata: 30% (~4x melhor que aleatório)

Com 14 classes, uma previsão aleatória daria ~7%. Os 30% mostram que a árvore aprendeu padrões reais — mas os números escondem um problema grave de **desequilíbrio de classes**.

Olhando para a **matriz de confusão**, o padrão é imediato: a coluna `drop_3` está repleta de valores em quase todas as linhas. A árvore aprendeu que `drop_3` é a resposta mais frequente (20619 amostras de 77101) e tende a prever isso para a maioria dos estados:

- **`drop_3`: 94% accuracy** — a classe dominante é bem prevista
- **`drop_2`, `drop_4`: 10-11%** — classes com bastantes amostras mas subrepresentadas face ao `drop_3`; a árvore classifica-as maioritariamente como `drop_3`
- **`drop_0`, `drop_1`, `drop_5`, `drop_6`: 0-3%** — classes laterais com menos amostras, quase sempre classificadas como `drop_3`
- **Todos os `pop_X`: 0%** — com apenas 175-446 amostras cada, a árvore nunca prevê pop. O desequilíbrio 33:1 torna estas classes invisíveis para o ID3

#### Raiz da árvore: `cell_0_3`

A primeira pergunta é o valor da célula na **linha 0 (topo), coluna 3 (central)**. Se `cell_0_3 == '.'`, a coluna está disponível e o MCTS escolhe `drop_3` na maioria dos casos. É a feature mais informativa do dataset — a disponibilidade da coluna central determina imediatamente o movimento mais provável.

#### A poda reduz 97% dos nós com custo mínimo

De 65991 para 1752 nós, mas a accuracy cai apenas de 30.11% para 29.58%. A árvore original estava a **overfittar massivamente** — criava nós terminais para combinações de células muito específicas que raramente aparecem no teste. A poda elimina-os com custo mínimo, resultando numa árvore muito mais compacta e generalizável.

#### Acurácia de tipo: 97% — o resultado mais importante

Em 11571 exemplos de teste, a árvore só erra o tipo (drop vs pop) em **350 casos**. A consistência no k-fold (95.90% ± 0.04%) confirma que este resultado é robusto.

Isto significa que a árvore aprendeu muito bem **quando fazer pop vs drop** — a decisão estrategicamente mais crítica, pois os pops são irreversíveis e alteram o tabuleiro de forma radical. A coluna exata é um erro mais tolerável do que o tipo errado.

#### Implicações como agente de jogo

O ID3 tem **comportamento estratégico razoável**: raramente comete o erro grave de fazer pop quando não devia. Mas não tem a precisão tática do MCTS na escolha da coluna ótima — vai frequentemente jogar na coluna central quando outra coluna seria melhor. O fallback para jogada aleatória válida garante que nunca fica bloqueado.

---
## 7. Modos de Jogo

O `PLAY.py` implementa três modos de jogo: Humano vs Humano, Humano vs MCTS, e MCTS vs ID3 automático.

**Como o ID3 é carregado — pickle:**

Treinar o ID3 com 77101 amostras demora horas. Para tornar o jogo instantâneo, separámos o treino do jogo em dois ficheiros. O `ID3_Treino.py` treina a árvore uma única vez com todas as amostras e guarda-a em `id3_tree.pkl` usando `pickle` — o módulo Python que serializa qualquer objeto para disco. Quando o `PLAY.py` arranca, carrega o pickle em menos de um segundo sem precisar de treinar novamente.

```python
# ID3_Treino.py — corre uma vez
tree = id3(X, y, list(X.columns))
with open('id3_tree.pkl', 'wb') as f:
    pickle.dump(tree, f)

# PLAY.py — carrega instantaneamente
with open('id3_tree.pkl', 'rb') as f:
    tree = pickle.load(f)
```

**Legality check:**

A função `id3_move` converte o estado do tabuleiro nas 43 features, percorre a árvore e obtém uma previsão. Como a árvore foi treinada com dados históricos, pode prever um movimento inválido naquele momento (por exemplo `drop_3` quando a coluna está cheia). O legality check verifica se o movimento previsto é legal — se não for, cai para uma jogada aleatória válida, garantindo que o ID3 nunca bloqueia.

In [ ]:
import pickle
from MCTS import MCTS
from PLAY import play_mcts_vs_id3, play_human_vs_human, play_human_vs_mcts

# Guardar a arvore ja treinada no pickle (so precisas de correr isto uma vez)
with open('id3_tree.pkl', 'wb') as f:
    pickle.dump(tree_popout, f)
print('id3_tree.pkl guardado.')

# Carregar do pickle — instantaneo em todas as execucoes seguintes
with open('id3_tree.pkl', 'rb') as f:
    tree_agent = pickle.load(f)

# MCTS vs ID3 — 100 jogos automaticos
results = play_mcts_vs_id3(tree_agent, n_games=100)

### 7.1 Resultados MCTS vs ID3 (100 jogos)

```
--- MCTS vs ID3 (100 jogos) ---

  Jogo   1  (começa: MCTS)  →  MCTS      Jogo  51  (começa: MCTS)  →  MCTS
  Jogo   2  (começa: ID3 )  →  ID3       Jogo  52  (começa: ID3 )  →  MCTS
  Jogo   3  (começa: MCTS)  →  MCTS      Jogo  53  (começa: MCTS)  →  MCTS
  Jogo   4  (começa: ID3 )  →  MCTS      Jogo  54  (começa: ID3 )  →  MCTS
  Jogo   5  (começa: MCTS)  →  MCTS      Jogo  55  (começa: MCTS)  →  MCTS
  Jogo   6  (começa: ID3 )  →  MCTS      Jogo  56  (começa: ID3 )  →  MCTS
  Jogo   7  (começa: MCTS)  →  MCTS      Jogo  57  (começa: MCTS)  →  MCTS
  Jogo   8  (começa: ID3 )  →  MCTS      Jogo  58  (começa: ID3 )  →  ID3
  Jogo   9  (começa: MCTS)  →  ID3       Jogo  59  (começa: MCTS)  →  MCTS
  Jogo  10  (começa: ID3 )  →  MCTS      Jogo  60  (começa: ID3 )  →  MCTS
  Jogo  11  (começa: MCTS)  →  MCTS      Jogo  61  (começa: MCTS)  →  MCTS
  Jogo  12  (começa: ID3 )  →  MCTS      Jogo  62  (começa: ID3 )  →  MCTS
  Jogo  13  (começa: MCTS)  →  MCTS      Jogo  63  (começa: MCTS)  →  MCTS
  Jogo  14  (começa: ID3 )  →  MCTS      Jogo  64  (começa: ID3 )  →  MCTS
  Jogo  15  (começa: MCTS)  →  ID3       Jogo  65  (começa: MCTS)  →  MCTS
  Jogo  16  (começa: ID3 )  →  MCTS      Jogo  66  (começa: ID3 )  →  MCTS
  Jogo  17  (começa: MCTS)  →  MCTS      Jogo  67  (começa: MCTS)  →  MCTS
  Jogo  18  (começa: ID3 )  →  MCTS      Jogo  68  (começa: ID3 )  →  MCTS
  Jogo  19  (começa: MCTS)  →  MCTS      Jogo  69  (começa: MCTS)  →  MCTS
  Jogo  20  (começa: ID3 )  →  MCTS      Jogo  70  (começa: ID3 )  →  MCTS
  Jogo  21  (começa: MCTS)  →  MCTS      Jogo  71  (começa: MCTS)  →  MCTS
  Jogo  22  (começa: ID3 )  →  MCTS      Jogo  72  (começa: ID3 )  →  ID3
  Jogo  23  (começa: MCTS)  →  MCTS      Jogo  73  (começa: MCTS)  →  ID3
  Jogo  24  (começa: ID3 )  →  MCTS      Jogo  74  (começa: ID3 )  →  MCTS
  Jogo  25  (começa: MCTS)  →  MCTS      Jogo  75  (começa: MCTS)  →  MCTS
  Jogo  26  (começa: ID3 )  →  MCTS      Jogo  76  (começa: ID3 )  →  MCTS
  Jogo  27  (começa: MCTS)  →  ID3       Jogo  77  (começa: MCTS)  →  MCTS
  Jogo  28  (começa: ID3 )  →  MCTS      Jogo  78  (começa: ID3 )  →  MCTS
  Jogo  29  (começa: MCTS)  →  ID3       Jogo  79  (começa: MCTS)  →  MCTS
  Jogo  30  (começa: ID3 )  →  ID3       Jogo  80  (começa: ID3 )  →  ID3
  Jogo  31  (começa: MCTS)  →  MCTS      Jogo  81  (começa: MCTS)  →  MCTS
  Jogo  32  (começa: ID3 )  →  MCTS      Jogo  82  (começa: ID3 )  →  ID3
  Jogo  33  (começa: MCTS)  →  MCTS      Jogo  83  (começa: MCTS)  →  MCTS
  Jogo  34  (começa: ID3 )  →  MCTS      Jogo  84  (começa: ID3 )  →  MCTS
  Jogo  35  (começa: MCTS)  →  MCTS      Jogo  85  (começa: MCTS)  →  MCTS
  Jogo  36  (começa: ID3 )  →  MCTS      Jogo  86  (começa: ID3 )  →  MCTS
  Jogo  37  (começa: MCTS)  →  ID3       Jogo  87  (começa: MCTS)  →  MCTS
  Jogo  38  (começa: ID3 )  →  ID3       Jogo  88  (começa: ID3 )  →  MCTS
  Jogo  39  (começa: MCTS)  →  MCTS      Jogo  89  (começa: MCTS)  →  MCTS
  Jogo  40  (começa: ID3 )  →  MCTS      Jogo  90  (começa: ID3 )  →  MCTS
  Jogo  41  (começa: MCTS)  →  MCTS      Jogo  91  (começa: MCTS)  →  MCTS
  Jogo  42  (começa: ID3 )  →  MCTS      Jogo  92  (começa: ID3 )  →  MCTS
  Jogo  43  (começa: MCTS)  →  MCTS      Jogo  93  (começa: MCTS)  →  MCTS
  Jogo  44  (começa: ID3 )  →  ID3       Jogo  94  (começa: ID3 )  →  MCTS
  Jogo  45  (começa: MCTS)  →  ID3       Jogo  95  (começa: MCTS)  →  MCTS
  Jogo  46  (começa: ID3 )  →  ID3       Jogo  96  (começa: ID3 )  →  MCTS
  Jogo  47  (começa: MCTS)  →  MCTS      Jogo  97  (começa: MCTS)  →  MCTS
  Jogo  48  (começa: ID3 )  →  MCTS      Jogo  98  (começa: ID3 )  →  MCTS
  Jogo  49  (começa: MCTS)  →  MCTS      Jogo  99  (começa: MCTS)  →  MCTS
  Jogo  50  (começa: ID3 )  →  MCTS      Jogo 100  (começa: ID3 )  →  MCTS

========================================
  RESULTADOS FINAIS (100 jogos)
========================================
  MCTS :  83  (83.0%)
  ID3  :  17  (17.0%)
  Draws:   0  (0.0%)

Acurácia por quem começa:
  MCTS começa primeiro : 47/50 vitórias MCTS  (94.0%)
  ID3  começa primeiro : 36/50 vitórias MCTS  (72.0%)
```

O MCTS venceu 83 dos 100 jogos, o ID3 venceu 17. Os 17% do ID3 são superiores ao que seria esperado de um agente puramente aleatório — confirmando que a árvore aprendeu padrões reais do jogo.
A diferença de performance é explicada pela natureza dos dois agentes: o MCTS raciocina dinamicamente com 300 simulações por jogada, enquanto o ID3 aplica uma função estática aprendida em treino.
O ID3 consegue ganhar quando o tabuleiro entra em configurações bem representadas no dataset, mas falha em posições menos comuns onde a previsão cai para o fallback aleatório.

In [ ]:
# Para jogar interativamente: descomentar e correr

# --- Humano vs Humano ---
# play_human_vs_human()

# --- Humano vs MCTS ---
# play_human_vs_mcts(human_player='X')

print('Para jogar, descomenta uma das opcoes acima e corre a celula.')

---
## 8. Conclusões

### MCTS

O torneio confirmou que o **UCB-V + Smart Rollout é a configuração mais forte**, com 78.7% de win-rate médio — venceu 76% dos jogos contra o UCT simples e 81% contra o UCT + Heurística. A razão é que as duas extensões se complementam: o smart rollout torna as simulações menos aleatórias e mais consistentes, o que reduz a variância dos resultados; o UCB-V aproveita exatamente essa redução de variância para calibrar melhor o termo de exploração. Juntos, fazem mais com o mesmo orçamento de simulações.

O UCT + Heurística ficou em último lugar apesar da heurística ser teoricamente boa — o problema foi temporal. Calcular o bónus posicional em cada nó aumentou o tempo por decisão para 1.685s, mais do triplo do UCT simples (0.445s). Com o mesmo número de iterações mas mais tempo por iteração, o agente explorava menos da árvore, e o resultado foi pior.

### Dataset

Em 8 horas gerámos 77101 amostras de 9267 jogos, a uma velocidade estável de ~1159 jogos/hora. O win-rate de 90.7% confirma que o MCTS era genuinamente bom como oráculo de treino. O desequilíbrio entre drops e pops (33:1) não é um erro de geração — é uma característica real do jogo, onde os pops só existem a partir do momento em que o jogador tem peças na base, e na maioria das posições um drop é estrategicamente superior.

### ID3 — Iris

O Iris serviu o seu propósito de warm-up. A árvore antes da poda tinha 14 nós e profundidade 4; depois da poda ficou com apenas 4 nós e profundidade 1, mantendo exatamente os mesmos 95.83% de accuracy. A MDL encontrou os cortes certos — `petal_length` separa quase perfeitamente as três classes — e a poda eliminou subárvores de overfitting sem custo. O k-fold de 96% ± 2.49% confirma que a implementação está correta e generaliza bem.

### ID3 — PopOut: o porquê dos 30%

O resultado de 30% de accuracy exata merece uma explicação cuidadosa porque à superfície parece mau, mas tem uma causa estrutural bem identificada. O problema é um ciclo vicioso entre o dataset e o treino:

O MCTS joga bem, por isso escolhe `drop_3` numa grande parte das posições (20619 de 77101 amostras, ou seja 27% do total). O ID3, ao treinar, aprende que `drop_3` é a melhor resposta para a maioria dos estados — e é mesmo, estatisticamente. Mas ao generalizar demasiado para `drop_3`, a árvore deixa de distinguir quando `drop_2` ou `drop_5` seriam melhores. Resultado: `drop_3` tem 94% de accuracy, todos os outros movimentos ficam entre 0% e 11%, e os pops ficam a 0% porque têm tão poucas amostras (175-446 cada) que a árvore nunca aprende a prever nenhum deles.

Dito de outra forma: o dataset reflete fielmente as preferências do MCTS, mas essas preferências têm uma distribuição muito concentrada que o ID3 não consegue modelar com uma única árvore sem dados mais equilibrados. O resultado não é um bug — é uma consequência direta de usar um oráculo forte (que tem preferências claras) para treinar um classificador flat.

A acurácia de tipo de 97% é a outra face da moeda: a decisão drop vs pop é aprendida quase perfeitamente porque depende de features muito concretas (presença de peças na linha base), e o k-fold confirma que isto é estável e robusto (±0.04%).

### Limitações e trabalho futuro

A hipótese mais promissora que não chegámos a testar seria dividir o ID3 em várias árvores especializadas — uma para situações de vitória imediata, uma para bloqueio, uma para jogo geral. Cada árvore teria classes mais equilibradas e decisões mais precisas, escapando ao ciclo de generalização excessiva para `drop_3`. O classificador de situação que decide qual árvore usar seria simples e poderia ele próprio ser uma pequena árvore ID3. Outra abordagem seria fazer oversampling das classes pop no dataset antes do treino, forçando o ID3 a aprender quando usar essas jogadas raras mas estrategicamente importantes.

---

## Autores

| Nome | Número |
|------|--------|
| Gonçalo Avó | 202405355 |
| Simão Moreira | 202407685 |
| Guilherme Xavier | 202406518 |